# SBS inline validation: OSAGO PD GLM Frequency

Ноутбук-шаблон для inline-прогона `M0-M5` по frequency GLM (`POISSON_GLM`) для датасета с `CLAIMS_PD_CNT`, `GLM_FREQ`, `EXPOSURE`.

Что важно:
- данные не запускаются автоматически, сначала укажи `DATA_PATH`;
- конфиг использует `score_column = "GLM_FREQ"`, `target_column = "CLAIMS_PD_CNT"`, `weight_column = "EXPOSURE"`;
- `IN_TRAIN_SPLIT_70_30_4_STRATIF_PD` нормализуется в `TRAIN`/`TEST`, если это 1/0 или boolean-индикатор;
- для GLM-пресета признаки в конфиге задаются базовыми именами: raw-категории идут в `categorical_features`, числовые/временные факторы — в `numeric_features`, а GLM-тесты сами выбирают `*_SCORING` или `*_VALUE` через resolver;
- `big_data_mode=True` включен по умолчанию, чтобы тяжелые тесты не пытались считать полный 17+ GB датасет.


## Настройки, сервисные колонки и списки факторов

Сначала задаем колонки и списки факторов, чтобы при чтении большого файла можно было загрузить только нужный subset колонок.


In [ ]:
from pathlib import Path
import os
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 180)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Данные лежат в другом месте: замени путь перед запуском.
DATA_PATH = Path(r"PUT_ABSOLUTE_PATH_TO_OSAGO_PD_GLM_FREQ_DATASET_HERE")
REPORTS_DIR = PROJECT_ROOT / "reports" / "osago_pd_glm_frequency"

# Для больших файлов оставь True: csv/parquet будут читаться только по нужным колонкам.
LOAD_REQUIRED_COLUMNS_ONLY = True

DATE_COLUMN = "ACC_QUARTER"
TARGET_COLUMN = "CLAIMS_PD_CNT"
SCORE_COLUMN = "GLM_FREQ"
WEIGHT_COLUMN = "EXPOSURE"
SAMPLE_COLUMN = "IN_TRAIN_SPLIT_70_30_4_STRATIF_PD"
ID_COLUMN = "UNIQUE_KEY"

WEIGHTED_SCORE_COLUMN = "GLM_FREQ_weighted"
TIME_COLUMNS_NOT_USED_AS_FEATURES = ["POLICY_START_DATE", "POLICY_END_DATE", "ACC_QUARTER", "POLICY_START_Y", "POLICY_START_Q", "POLICY_START_M"]
PREDICTION_BIN_COLUMNS = ["GLM_MODEL_BINQ10", "GLM_MODEL_weighted_BINQ10"]
INTERCEPT_COLUMNS = ["INTERCEPT", "INTERCEPT_VALUE"]

raw_numeric_features = """
POLICY_SIGN_START_DIFF
POLICY_LDU_WORST_AGE
POLICY_LDU_WORST_EXP
POLICY_CALC_COUNT_90
VEHICLE_RTA_CNT_OWNER_ALL_TIME_OBSTACLE_CNT
""".split()

raw_categorical_features = """
POLICY_FRANCHISE_TYPE
POLICY_FRANCHISE_SUM_GR_23_MODEL
POLICY_SI_GR
POLICY_LDU_WORST_KBM
POLICY_LDU_WORST_NO_ACCIDENT_EXP
POLICY_CAR_MARK
POLICY_CAR_AGE_2
VEHICLE_MKT_ENCODING_BRAND_MODEL_V1_A6_W100
""".split()

model_numeric_features = [*raw_numeric_features, "POLICY_START_Y"]
configured_glm_feature_bases = [*model_numeric_features, *raw_categorical_features]

provided_numeric_companion_columns = """
POLICY_CALC_COUNT_90_SCORING
POLICY_CAR_AGE_2_SCORING
POLICY_CAR_MARK_SCORING
POLICY_FRANCHISE_SUM_GR_23_MODEL_SCORING
POLICY_FRANCHISE_TYPE_SCORING
POLICY_LDU_WORST_AGE_SCORING
POLICY_LDU_WORST_EXP_SCORING
POLICY_LDU_WORST_KBM_SCORING
POLICY_LDU_WORST_NO_ACCIDENT_EXP_SCORING
POLICY_SI_GR_SCORING
POLICY_SIGN_START_DIFF_SCORING
POLICY_START_Y_SCORING
VEHICLE_MKT_ENCODING_BRAND_MODEL_V1_A6_W100_SCORING
VEHICLE_RTA_CNT_OWNER_ALL_TIME_OBSTACLE_CNT_SCORING
POLICY_CALC_COUNT_90_VALUE
POLICY_CAR_AGE_2_VALUE
POLICY_CAR_MARK_VALUE
POLICY_FRANCHISE_SUM_GR_23_MODEL_VALUE
POLICY_FRANCHISE_TYPE_VALUE
POLICY_LDU_WORST_AGE_VALUE
POLICY_LDU_WORST_EXP_VALUE
POLICY_LDU_WORST_KBM_VALUE
POLICY_LDU_WORST_NO_ACCIDENT_EXP_VALUE
POLICY_SI_GR_VALUE
POLICY_SIGN_START_DIFF_VALUE
POLICY_START_Y_VALUE
VEHICLE_MKT_ENCODING_BRAND_MODEL_V1_A6_W100_VALUE
VEHICLE_RTA_CNT_OWNER_ALL_TIME_OBSTACLE_CNT_VALUE
""".split()

provided_scoring_columns = [c for c in provided_numeric_companion_columns if c.endswith("_SCORING")]
provided_value_columns = [c for c in provided_numeric_companion_columns if c.endswith("_VALUE")]
glm_feature_bases = [c.removesuffix("_SCORING") for c in provided_scoring_columns]
expected_value_columns = [f"{base}_VALUE" for base in glm_feature_bases]

required_columns = [DATE_COLUMN, TARGET_COLUMN, SCORE_COLUMN, WEIGHT_COLUMN, SAMPLE_COLUMN, ID_COLUMN]
optional_diagnostic_columns = [WEIGHTED_SCORE_COLUMN, *PREDICTION_BIN_COLUMNS, *INTERCEPT_COLUMNS]
required_data_columns = list(dict.fromkeys([
    *required_columns,
    *raw_numeric_features,
    *raw_categorical_features,
    *glm_feature_bases,
    *provided_numeric_companion_columns,
    *optional_diagnostic_columns,
]))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"REPORTS_DIR: {REPORTS_DIR}")
print(f"Data file exists: {DATA_PATH.exists()}")
print(f"Columns requested from file: {len(required_data_columns)}")


## Загрузка данных

Ячейка поддерживает `csv`, `parquet`, `xlsx/xls` и `pickle/pkl`. Для `csv` и `parquet` при `LOAD_REQUIRED_COLUMNS_ONLY=True` читаются только нужные колонки.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Укажи реальный DATA_PATH перед запуском ноутбука. Сейчас стоит placeholder."
    )

suffix = DATA_PATH.suffix.lower()
if suffix == ".csv":
    if LOAD_REQUIRED_COLUMNS_ONLY:
        data = pd.read_csv(DATA_PATH, usecols=lambda c: c in set(required_data_columns))
    else:
        data = pd.read_csv(DATA_PATH)
elif suffix in {".parquet", ".pq"}:
    if LOAD_REQUIRED_COLUMNS_ONLY:
        data = pd.read_parquet(DATA_PATH, columns=required_data_columns)
    else:
        data = pd.read_parquet(DATA_PATH)
elif suffix in {".xlsx", ".xls"}:
    data = pd.read_excel(DATA_PATH)
    if LOAD_REQUIRED_COLUMNS_ONLY:
        data = data[[c for c in required_data_columns if c in data.columns]].copy()
elif suffix in {".pickle", ".pkl"}:
    data = pd.read_pickle(DATA_PATH)
    if LOAD_REQUIRED_COLUMNS_ONLY:
        data = data[[c for c in required_data_columns if c in data.columns]].copy()
else:
    raise ValueError(f"Unsupported data file suffix: {suffix}")

print(f"Data shape: {data.shape[0]:,} rows x {data.shape[1]:,} columns")
print(f"DataFrame memory: {data.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
display(data.head())


## Контроль состава признаков

В `config.columns` кладем базовые имена GLM-факторов без суффиксов. Raw-факторы явно разделены на numeric/categorical; `POLICY_START_Y` добавлен только как model-level numeric base, чтобы его `*_SCORING`/`*_VALUE` участвовали в GLM-тестах. Для GLM-тестов физические `*_SCORING` и `*_VALUE` выбираются автоматически.


In [ ]:
missing_value_columns_in_list = sorted(set(expected_value_columns) - set(provided_value_columns))
extra_value_columns_in_list = sorted(set(provided_value_columns) - set(expected_value_columns))
configured_not_in_glm_bases = sorted(set(configured_glm_feature_bases) - set(glm_feature_bases))
glm_bases_not_configured = sorted(set(glm_feature_bases) - set(configured_glm_feature_bases))

print("Service columns:")
print({
    "date_column": DATE_COLUMN,
    "target_column": TARGET_COLUMN,
    "score_column": SCORE_COLUMN,
    "weight_column": WEIGHT_COLUMN,
    "sample_column": SAMPLE_COLUMN,
    "id_column": ID_COLUMN,
})

display(pd.DataFrame({
    "feature_base": pd.Series(glm_feature_bases),
    "scoring_column": pd.Series(provided_scoring_columns),
    "expected_value_column": pd.Series(expected_value_columns),
    "used_in_config": pd.Series([base in set(configured_glm_feature_bases) for base in glm_feature_bases]),
}))

print(f"Raw numeric reference columns: {len(raw_numeric_features)}")
print(f"Model numeric feature bases: {len(model_numeric_features)}")
print(f"Raw categorical reference columns: {len(raw_categorical_features)}")
print(f"Provided *_SCORING columns: {len(provided_scoring_columns)}")
print(f"Provided *_VALUE columns: {len(provided_value_columns)}")
print(f"Configured GLM feature bases: {len(configured_glm_feature_bases)}")
print(f"Missing VALUE columns in provided list: {missing_value_columns_in_list}")
print(f"Extra VALUE columns in provided list: {extra_value_columns_in_list}")
print(f"Configured columns not present as GLM bases: {configured_not_in_glm_bases}")
print(f"GLM bases not used in config: {glm_bases_not_configured}")


## Подготовка датафрейма

Здесь приводим типы, парсим квартальную дату, нормализуем sample и создаем aliases для companion-only факторов. Сервисные колонки, бины предсказания и intercept не добавляются в список фичей.


In [ ]:
def parse_quarter_or_date(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(series, errors="coerce")
    if parsed.notna().mean() >= 0.95:
        return parsed

    text = series.astype(str).str.strip()
    quarters = text.str.extract(r"(?P<year>\d{4})\D*Q(?P<quarter>[1-4])", expand=True)
    out = parsed.copy()
    mask = quarters["year"].notna() & quarters["quarter"].notna()
    if mask.any():
        periods = pd.PeriodIndex(
            quarters.loc[mask, "year"] + "Q" + quarters.loc[mask, "quarter"],
            freq="Q",
        )
        out.loc[mask] = periods.to_timestamp()
    if out.isna().any():
        bad_examples = series[out.isna()].head(10).tolist()
        raise ValueError(f"Не удалось распарсить {DATE_COLUMN}; examples={bad_examples}")
    return out


def normalize_sample_split(series: pd.Series) -> pd.Series:
    """Normalize train/test split to labels accepted by SBS validation.

    The source column is named like a train indicator, so 1/true -> TRAIN and
    0/false -> TEST. Existing TRAIN/TEST/OOS/OOT labels are preserved.
    """
    normalized = series.astype(str).str.strip().str.lower().str.replace(" ", "_", regex=False)
    train_values = {"1", "1.0", "true", "t", "yes", "y", "train", "training", "development", "dev"}
    test_values = {"0", "0.0", "false", "f", "no", "n", "test", "oos", "validation", "valid", "val"}
    oot_values = {"oot", "out_of_time", "out-of-time", "outoftime"}

    out = normalized.copy()
    out = out.mask(normalized.isin(train_values), "TRAIN")
    out = out.mask(normalized.isin(test_values), "TEST")
    out = out.mask(normalized.isin(oot_values), "OOT")
    out = out.str.upper()
    return out


final_df = data.copy()

missing_required_columns = [c for c in required_columns if c not in final_df.columns]
if missing_required_columns:
    raise KeyError(f"В данных нет обязательных колонок: {missing_required_columns}")

final_df[DATE_COLUMN] = parse_quarter_or_date(final_df[DATE_COLUMN])
final_df[TARGET_COLUMN] = pd.to_numeric(final_df[TARGET_COLUMN], errors="raise")
final_df[SCORE_COLUMN] = pd.to_numeric(final_df[SCORE_COLUMN], errors="raise")
final_df[WEIGHT_COLUMN] = pd.to_numeric(final_df[WEIGHT_COLUMN], errors="raise")
final_df[ID_COLUMN] = final_df[ID_COLUMN].astype(str)
final_df[SAMPLE_COLUMN] = normalize_sample_split(final_df[SAMPLE_COLUMN])

for column in provided_numeric_companion_columns:
    if column in final_df.columns:
        final_df[column] = pd.to_numeric(final_df[column], errors="raise")

if WEIGHTED_SCORE_COLUMN not in final_df.columns:
    final_df[WEIGHTED_SCORE_COLUMN] = final_df[SCORE_COLUMN] * final_df[WEIGHT_COLUMN]

missing_base_columns = [
    base for base in glm_feature_bases
    if base not in final_df.columns and f"{base}_SCORING" in final_df.columns
]
for base in missing_base_columns:
    final_df[base] = final_df[f"{base}_SCORING"]

missing_scoring_in_data = [c for c in provided_scoring_columns if c not in final_df.columns]
missing_value_in_data = [c for c in expected_value_columns if c not in final_df.columns]
missing_base_after_alias = [base for base in glm_feature_bases if base not in final_df.columns]

print(f"Missing base columns created as aliases: {len(missing_base_columns)}")
print(f"Missing *_SCORING columns in data: {len(missing_scoring_in_data)}")
print(f"Missing *_VALUE columns in data: {len(missing_value_in_data)}")
print(f"Missing base columns after aliasing: {missing_base_after_alias}")

if missing_scoring_in_data or missing_value_in_data or missing_base_after_alias:
    print("WARNING: проверь missing lists выше. Для GLM M3.1/M3.2 особенно нужны *_VALUE columns.")

split_summary = final_df[SAMPLE_COLUMN].value_counts(dropna=False).rename_axis(SAMPLE_COLUMN).reset_index(name="rows")
date_summary = pd.DataFrame({
    "metric": ["min_date", "max_date", "missing_dates", "bad_target_negative", "bad_exposure_nonpositive"],
    "value": [
        final_df[DATE_COLUMN].min(),
        final_df[DATE_COLUMN].max(),
        final_df[DATE_COLUMN].isna().sum(),
        int((final_df[TARGET_COLUMN] < 0).sum()),
        int((final_df[WEIGHT_COLUMN] <= 0).sum()),
    ],
})

display(split_summary)
display(date_summary)


## SBS config

Для `POISSON_GLM` вес `EXPOSURE` обязателен и трактуется как exposure. `numeric_features` и `categorical_features` здесь — явно заданные базовые GLM-факторы, разделенные по raw-смыслу признака.


In [ ]:
from sbs_evaluation_tool import POISSON_GLM
from sbs_evaluation_tool.core.config_validation import validate_config

numeric_features_for_config = [
    feature for feature in model_numeric_features
    if feature in set(glm_feature_bases)
]
categorical_features_for_config = [
    feature for feature in raw_categorical_features
    if feature in set(glm_feature_bases)
]

config = {
    "preset": POISSON_GLM,
    "validation_mode": "oos",
    "performance": {
        "big_data_mode": True,
    },
    "columns": {
        "numeric_features": numeric_features_for_config,
        "categorical_features": categorical_features_for_config,
        "score_column": SCORE_COLUMN,
        "date_column": DATE_COLUMN,
        "target_column": TARGET_COLUMN,
        "sample_column": SAMPLE_COLUMN,
        "id_column": ID_COLUMN,
        "weight_column": WEIGHT_COLUMN,
    },
}

validated_config = validate_config(config, final_df)
print(validated_config)
print(f"Numeric features in config: {len(validated_config.columns.numeric_features)}")
print(f"Categorical features in config: {len(validated_config.columns.categorical_features)}")
print(f"big_data_mode: {validated_config.performance.big_data_mode}")


## Inline SBS tests: полный набор M0-M5

Ниже каждая группа запускается отдельной ячейкой. Это удобно для тяжелого отчета: можно перезапускать только нужный блок и смотреть результат прямо в ноутбуке.


In [ ]:
from sbs_evaluation_tool.core.inline_display import show_test_inline
from sbs_evaluation_tool.tests.psi_calculator import PSICalculatorExtended

psi_calc = PSICalculatorExtended()
# В тесты передаем raw dict config: часть legacy-кода еще вызывает config.get(...).
# validated_config выше нужен как явная проверка, что конфиг корректен.
common_params = {"df": final_df, "config": config}


### M0. Data Quality

In [ ]:
from sbs_evaluation_tool.tests.m0_data_quality import (
    M01_DataQualityOverviewTest,
    M04_MonthlyDistributionsTest,
    M05_ModeAndMissingByPeriodTest,
)

show_test_inline(M01_DataQualityOverviewTest(), common_params)

# M0.4 строит exploratory monthly distributions по полному согласованному raw/base набору:
# raw-категории отдельно, raw-числовые факторы отдельно.
m04_categorical_features = [c for c in raw_categorical_features if c in final_df.columns]
m04_numeric_features = [c for c in raw_numeric_features if c in final_df.columns]

print(f"M0.4 categorical features: {len(m04_categorical_features)}")
print(f"M0.4 numeric features: {len(m04_numeric_features)}")

show_test_inline(
    M04_MonthlyDistributionsTest(),
    {
        **common_params,
        "categorical_features": m04_categorical_features,
        "numeric_features": m04_numeric_features,
    },
)
show_test_inline(
    M05_ModeAndMissingByPeriodTest(),
    {
        **common_params,
        "categorical_features": m04_categorical_features,
        "numeric_features": m04_numeric_features,
    },
)


### M1. Distribution / PSI

С текущей библиотекой M1 берет признаки напрямую из `config.columns`, поэтому для этих двух тестов временно подменяем feature-list в том же `config` на физические `*_SCORING`-колонки и сразу восстанавливаем base-факторы после прогона.

In [ ]:
from sbs_evaluation_tool.tests.m1_distribution_tests import (
    M11_TrainVsGenpopPSITest,
    M12_OOSVsGenpopPSITest,
    M13_MaxDateRecencyTest,
)

original_numeric_features = list(config["columns"]["numeric_features"])
original_categorical_features = list(config["columns"]["categorical_features"])

try:
    m1_scoring_columns = [
        f"{base}_SCORING"
        for base in [*numeric_features_for_config, *categorical_features_for_config]
        if f"{base}_SCORING" in final_df.columns
    ]
    config["columns"]["numeric_features"] = m1_scoring_columns
    config["columns"]["categorical_features"] = []
    print(f"M1 PSI features (*_SCORING): {len(config['columns']['numeric_features'])}")
    show_test_inline(M11_TrainVsGenpopPSITest(psi_calc=psi_calc), common_params)
    show_test_inline(M12_OOSVsGenpopPSITest(psi_calc=psi_calc), common_params)
finally:
    config["columns"]["numeric_features"] = original_numeric_features
    config["columns"]["categorical_features"] = original_categorical_features

show_test_inline(M13_MaxDateRecencyTest(), common_params)


### M2. Gini

In [ ]:
from sbs_evaluation_tool.tests.m2_gini_tests import (
    M21_GiniTest,
    M22_GiniFactorsTest,
    M23_GiniDynamicsTest,
    M24_GiniUpliftTest,
)

show_test_inline(M21_GiniTest(), common_params, render="plotly")
show_test_inline(M22_GiniFactorsTest(), common_params)
show_test_inline(M23_GiniDynamicsTest(), common_params, render="plotly")
show_test_inline(M24_GiniUpliftTest(), common_params)


### M3. Multicollinearity

In [ ]:
from sbs_evaluation_tool.tests.m3_multicollinearity import (
    M31_LikelihoodRatioTest,
    M32_MulticollinearityTest,
)

show_test_inline(M31_LikelihoodRatioTest(), common_params)
show_test_inline(M32_MulticollinearityTest(), common_params)


### M4. Calibration

In [ ]:
from sbs_evaluation_tool.tests.m4_model_performance import (
    M41_TargetRateTest,
    M42_CalibrationCurveByPredictBinsLn,
    M43_CalibrationCurveByDatesLn,
)

show_test_inline(M41_TargetRateTest(), common_params)
show_test_inline(
    M42_CalibrationCurveByPredictBinsLn(),
    {**common_params, "num_groups": 20},
    render="plotly",
)
show_test_inline(M43_CalibrationCurveByDatesLn(), common_params, render="plotly")


### M5. Stability

In [ ]:
from sbs_evaluation_tool.tests.m5_model_stability import (
    M51_ModelGiniStabilityTest,
    M52_FactorGiniStabilityTest,
    M53_ScorePSITest,
    M54_FactorPSITest,
)

show_test_inline(M51_ModelGiniStabilityTest(), common_params)
show_test_inline(M52_FactorGiniStabilityTest(), common_params)
show_test_inline(M53_ScorePSITest(psi_calc=psi_calc), common_params)
show_test_inline(M54_FactorPSITest(psi_calc=psi_calc), common_params)


## Optional full HTML report

Inline-ячейки выше — основной сценарий для ноутбука. Полный HTML можно включить вручную, если нужен отдельный файл отчета.


In [ ]:
RUN_FULL_REPORT = False

if RUN_FULL_REPORT:
    from sbs_evaluation_tool.core.runner import run_sbs_evaluation
    from IPython.display import HTML, display

    report_path = run_sbs_evaluation(
        data=final_df,
        config=config,
        segment_name="OSAGO_PD_GLM_Frequency",
        output_dir=str(REPORTS_DIR),
        size="normal",
    )

    report_file = Path(report_path)
    print(f"Report exists: {report_file.exists()}")
    print(f"Report size: {report_file.stat().st_size if report_file.exists() else 0:,} bytes")
    print(f"Report path: {report_file}")

    if report_file.exists():
        display(HTML(f'<a href="{report_file.as_uri()}" target="_blank">Open SBS HTML report</a>'))
else:
    print("Full HTML report generation skipped. Set RUN_FULL_REPORT = True to run M0-M5 into HTML.")
